# Research workflow — offline: question to exact economics

Owner: repository founder. Updated 2026-09-21. Demonstrates features, models, exact economics and bounded rule comparisons.

Follow **question → pinned data → quality → features and labels → retained model comparison
→ rule targets → independent risk → delayed fills → exact ledger → developer handoff**.
The first lesson keeps the existing mixed-input inspection. A separate, explicitly synthetic
trade study then supplies known availability for the connected feature/model/economic path.
It never repairs or relabels an unknown real archive's history.

Restart and **Run All** using the [operations guide](notebook-operations.md). Defaults make no
provider or tracking request. Every run retains inputs, configurations and attempts under
ignored `examples/runs/`; copy the complete artifact directory for durable retention.
An explicitly supplied saved selection is read without fallback; the later synthetic study
remains separate and cannot establish a result for that selection.

The [developer notebook](infrastructure-diagnostics.ipynb) exercises installed consumers,
fresh-process recovery and failure boundaries. The [live version](research-foundations-live.ipynb)
collects book data only. Neither route establishes empirical alpha, liquidity, frozen P1
acceptance, deployment or trading authority. Configured reports and serial pipelines are available; MR 7 remains partial pending Evidently admission.

## 1. Frame the research questions

First inspect whether the selected book can support a visible-depth prototype. Then ask a
separate software question: can one saved trade study reproduce exact features, delayed
labels, a train-only model and a costed fixed rule? The model and trading rule remain
separate consumers: MR 5 does not bind fitted predictions to executable targets.
Declare the comparator and stopping criteria before seeing outputs. No final holdout is read.

In [1]:
question = {
    "inspection": "Can the selected book support a visible-depth diagnostic?",
    "connected_study": "Do retained price diagnostics and costed fixed rules reproduce?",
    "sample": "tiny synthetic inputs; any supplied saved selection stays separate",
    "comparators": "zero predictor versus trained least squares; no-trade versus momentum",
    "stop_if": "clock, units, retained input, split, valuation or accounting contracts fail",
    "holdout": "reserved future interval; never queried or consumed",
    "acceptance": "software reconciliation, not profitability or model quality",
}
question

{'inspection': 'Can the selected book support a visible-depth diagnostic?',
 'connected_study': 'Do retained price diagnostics and costed fixed rules reproduce?',
 'sample': 'tiny synthetic inputs; any supplied saved selection stays separate',
 'comparators': 'zero predictor versus trained least squares; no-trade versus momentum',
 'stop_if': 'clock, units, retained input, split, valuation or accounting contracts fail',
 'holdout': 'reserved future interval; never queried or consumed',
 'acceptance': 'software reconciliation, not profitability or model quality'}

### Prepare the installed candidate

The complete runtime file inventory must match the checkout. A package version alone
cannot identify changed source. This check records the interpreter, lock, source digest
and notebook source digest; the runner additionally preserves each execution attempt.
Expand setup cells if investigating an installation mismatch.


In [2]:
# standard library imports
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from decimal import Decimal
from importlib.metadata import distributions
from pathlib import Path
from uuid import uuid4

# third party library imports
import pandas as pd
from IPython.display import display
from msgspec.structs import replace

# project library imports
import market_data
from applications.explore import inspect_eda
from market_data.adapters.sources import SourceAdapters
from market_data.contracts import Level
from market_data.datasets import SnapshotCatalog
from market_data.query import BookMarketRow, QueryMarketRow
from market_data.storage import RawStore
from research.exploration import (
    BookResearchInput,
    ResearchDataset,
    ResearchReader,
    ResearchSelection,
    TradeResearchInput,
    TradeSelection,
)
from runtime import ImplementationIdentity, RuntimeFiles
from serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/data-jobs/binance-archive.json").is_file(), "Use the repository kernel/root"
assert platform.python_version() == (root / ".python-version").read_text().strip()
assert "site-packages" in Path(market_data.__file__).resolve().parts
pd.set_option("display.max_colwidth", None)
codec = DomainCodec()
attempt_key = os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", uuid4().hex)
local = root / "examples/runs/research-foundations" / f"{attempt_key}-artifacts"
local.mkdir(parents=True, exist_ok=False)
(local / "question.json").write_bytes(codec.encode_json(question))

549

In [3]:
source_root = root / "src"
installed_inputs = dict(RuntimeFiles.inputs())
source_files = {
    path.relative_to(source_root).as_posix(): path
    for package in RuntimeFiles.roots
    for path in (source_root / package).rglob("*")
    if path.is_file() and (path.suffix in {".py", ".json"} or path.name == "py.typed")
}
assert source_files.keys() == installed_inputs.keys(), "Installed runtime inventory differs"
for name, source in source_files.items():
    assert source.read_bytes() == installed_inputs[name].read_bytes(), (
        f"Stale installed file: {name}"
    )
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations.ipynb")
)
notebook = json.loads(notebook_path.read_text())
notebook_source = [(cell["cell_type"], "".join(cell["source"])) for cell in notebook["cells"]]
identity = {
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip(),
    "working_tree": "changed"
    if subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
    else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel", "pandas"}
    },
    "kernel": sys.executable,
    "package": market_data.__file__,
    "runtime_files": len(installed_inputs),
    "implementation_sha256": ImplementationIdentity.sha256(),
    "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "notebook_source_sha256": hashlib.sha256(
        json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
    ).hexdigest(),
    "attempt": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
}
display(pd.DataFrame(identity.items(), columns=["Identity", "Value"]))

,Identity,Value
0,commit,c2b140e86e431b6c5c770c24aa3b9ef3e70f7864
1,working_tree,changed
2,python,3.14.7
3,platform,macOS-27.0-arm64-arm-64bit-Mach-O
4,tooling,"{'nbformat': '5.10.4', 'ipykernel': '7.2.0', 'nbclient': '0.10.4', 'pandas': '3.0.5'}"
5,kernel,/Users/lxg/.cache/uv/builds-v0/.tmpOD9nXF/bin/python
6,package,/Users/lxg/.codex/worktrees/mr-07-research-comparison/alpha-research-lab-private/.venv/lib/python3.14/site-packages/market_data/__init__.py
7,runtime_files,190
8,implementation_sha256,a3b2511665cc4f93af87a31760a050120b4928a1adc352b6629ff1f4af8b22e5
9,lock_sha256,b3c7e6cae180a590f5bbb8cbe09106325ba83de8fb0ad7f916f64947269ae138


## 2. Select or onboard the required data

For an existing study, set `ALPHA_RESEARCH_DATASET_DIRECTORY` to a retained directory
containing `raw/`, `snapshots/` and an authored `selection.json`. Both stores must retain
the bytes addressed by that selection. The notebook reads them and writes its own reports
to a separate retained artifact directory. No mutable latest-input choice is made.

With no override, the collapsed setup calls the existing
[installed acquisition/publication consumer](../tests/integration/consumers/datasets/ingestion_workflow.py).
It supplies synthetic checksummed HTTP responses for the configured `spot-comparison`
job and a synthetic native book. Package acquisition and publication own the mechanics;
the notebook starts from their saved stores. This also exercises their failure paths,
whose engineering interpretation belongs in the diagnostic notebook.

To obtain new permitted data, use the existing [data commands](../src/market_data/README.md)
and [capture instructions](notebook-operations.md#acquire-and-reopen-permitted-inputs).
A download supplies bytes; it does not establish historical availability or eligibility.


In [4]:
override = os.environ.get("ALPHA_RESEARCH_DATASET_DIRECTORY")
synthetic = override is None
if synthetic:
    fixture_inputs = local / "inputs"
    fixture_inputs.mkdir()
    for relative in (
        "tests/integration/consumers/datasets/ingestion_workflow.py",
        "tests/support/subprocesses/interrupt_command.py",
        "tests/fixtures/eligibility/membership-corrections.json",
        "configs/data-jobs/binance-archive.json",
        "configs/live/okx-btc-usdt.json",
        "configs/reference/binance-archive-spot.json",
    ):
        source = root / relative
        shutil.copyfile(source, fixture_inputs / source.name)
    data_directory = local / "data"
    prepared = subprocess.run(
        [
            sys.executable,
            "-I",
            str(fixture_inputs / "ingestion_workflow.py"),
            str(fixture_inputs),
            str(data_directory),
        ],
        cwd=local,
        capture_output=True,
        text=True,
        timeout=60,
        check=False,
    )
    assert prepared.returncode == 0, (prepared.stdout, prepared.stderr)
else:
    data_directory = Path(override).expanduser().resolve()
if not all((data_directory / name).is_dir() for name in ("raw", "snapshots")):
    raise FileNotFoundError(f"Saved study requires raw/ and snapshots/: {data_directory}")
saved_selection = codec.decode_json(
    (data_directory / "selection.json").read_bytes(), ResearchSelection
)
print("Input mode:", "synthetic software example" if synthetic else "explicit saved selection")
print("Retained input directory:", data_directory)
print("Saved selection identity:", saved_selection.identity)

Input mode: synthetic software example
Retained input directory: /Users/lxg/.codex/worktrees/mr-07-research-comparison/alpha-research-lab-private/examples/runs/research-foundations/dfd2e123-d034-4e5d-8c6d-a012fc764dcc-artifacts/data
Saved selection identity: 0a6c78bc4b2d4fc22f0d8a7f6a91d40cc5427362d31b487c77eac4876f6433a1


### Discover metadata, then pin the study

Discovery is bounded metadata inspection. Opening the selection later verifies all
required raw and canonical inputs. Choose the named bindings, immutable snapshot IDs,
fields, intervals and one shared inclusive knowledge cutoff explicitly. Trade intervals
use business time; book intervals use receipt time; both exclude the end.

Edit `selection` below by constructing or replacing the public selection records, using
[the reference](../src/market_data/README.md#discover-and-reopen-research-selections).
Keeping the loaded records is an explicit pin to their saved identities, not to the
newest snapshot. Change byte/entry caps only for the intended workload; batch size does
not cap total process memory.


In [5]:
candidates = SnapshotCatalog(data_directory / "snapshots").discover(
    max_entries=64, max_bytes=2_097_152
)
display(
    pd.DataFrame(
        [
            {
                "snapshot": item.identity,
                "family": type(item).__name__,
                "partitions": len(item.partitions),
                "quality_scope": item.scope,
            }
            for item in candidates
        ]
    )
)
selection = ResearchSelection(inputs=saved_selection.inputs)
selection_path = local / "selection.json"
selection_path.write_bytes(codec.encode_json(selection))
display(
    pd.DataFrame(
        [
            {
                "name": item.name,
                "family": "trade" if isinstance(item, TradeSelection) else "book",
                "snapshot": item.query.snapshot_id,
                "fields": item.query.columns,
                "instruments": tuple(map(str, item.query.instruments))
                if isinstance(item, TradeSelection)
                else (str(item.query.instrument),),
                "start": item.query.start,
                "end_excluded": item.query.end,
                "known_at_inclusive": item.query.known_at,
                "byte_cap": item.query.max_bytes,
            }
            for item in selection.inputs
        ]
    )
)
print("Pinned selection:", selection.identity)

,snapshot,family,partitions,quality_scope
0,855120b096dde030de79159c22aa76aba3cbe0334e0f8049eb2673b0edc040d3,BookDatasetManifest,1,exploratory_degraded
1,c14dc08bd794fb0a4c17c07a909857aa02c1bb35a4ea05224dc03e761e4b9b6d,DatasetManifest,2,exploratory_degraded


,name,family,snapshot,fields,instruments,start,end_excluded,known_at_inclusive,byte_cap
0,btc,trade,c14dc08bd794fb0a4c17c07a909857aa02c1bb35a4ea05224dc03e761e4b9b6d,"(price, revision)","(11111111-1111-4111-8111-111111111111,)",2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,2026-09-18 00:00:00+00:00,262144
1,eth,trade,c14dc08bd794fb0a4c17c07a909857aa02c1bb35a4ea05224dc03e761e4b9b6d,"(price, revision)","(22222222-2222-4222-8222-222222222222,)",2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,2026-09-18 00:00:00+00:00,262144
2,okx_depth,book,855120b096dde030de79159c22aa76aba3cbe0334e0f8049eb2673b0edc040d3,"(bids, asks)","(4476df8e-6342-4417-839e-e41b88f41446,)",2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,2026-09-18 00:00:00+00:00,262144


Pinned selection: 0a6c78bc4b2d4fc22f0d8a7f6a91d40cc5427362d31b487c77eac4876f6433a1


## 3. Inspect values, missingness and clocks together

The existing EDA entry retains started, completed and failed records through a caller-owned
callback. It returns separate typed inputs, without implicit joins, currency conversion or
forward filling. Inspect coverage and control rows even when no market values are selected.
Unknown archive availability should produce unavailable values, never fabricated timestamps.


In [6]:
class StudyReports:
    """Append the small EDA attempt records in this notebook's private workspace."""

    @staticmethod
    def retain(record: dict[str, object]) -> None:
        """Save one started, completed or failed application record.

        Args:
            record: Application-owned identity, status and evidence.
        """
        with (local / "eda-attempts.jsonl").open("ab") as output:
            output.write(codec.encode_json(record) + b"\n")


result = inspect_eda(
    selection_path,
    data_directory / "snapshots",
    RawStore(data_directory / "raw"),
    report=StudyReports.retain,
)
dataset = result["dataset"]
assert isinstance(dataset, ResearchDataset) and dataset.identity == selection.identity
observations = []
for item in dataset.inputs:
    if isinstance(item, TradeResearchInput):
        evidence = item.batches[0].evidence
        count = sum(isinstance(row, QueryMarketRow) for batch in item.batches for row in batch.rows)
        units = tuple(str(part.request.instrument.quote_currency.code) for part in evidence)
        reasons = sorted({reason for coverage in item.coverage for reason in coverage.reasons})
    else:
        count = sum(isinstance(row, BookMarketRow) for batch in item.batches for row in batch.rows)
        config = item.batches[0].evidence.request.config
        units = (config.quote_currency,)
        reasons = sorted({reason for batch in item.batches for reason in batch.reasons})
    total = sum(len(batch.rows) for batch in item.batches)
    observations.append(
        {
            "input": item.name,
            "market_rows": count,
            "controls": total - count,
            "quote_currency": units,
            "reasons": reasons,
        }
    )
display(pd.DataFrame(observations))
if synthetic:
    assert [row["market_rows"] for row in observations] == [0, 0, 1]

,input,market_rows,controls,quote_currency,reasons
0,btc,0,0,"(USDT,)","[catalog units are authored research assumptions, no selected trade values; inspect availability and membership evidence, observed values do not establish complete coverage, original trade availability unknown, recorder coverage unknown]"
1,eth,0,0,"(USDT,)","[catalog units are authored research assumptions, no selected trade values; inspect availability and membership evidence, observed values do not establish complete coverage, original trade availability unknown, recorder coverage unknown]"
2,okx_depth,1,3,"(USDT,)",[visible depth only; coverage and historical eligibility unproven]


The default trades contain six retained historical observations but **zero decision-eligible
trade values**: archive download time is not original availability. The one book market
row has an actual synthetic receipt coordinate. Controls remain separate from values.
Coverage and historical eligibility remain unproven even for the observed book state.

**Exercise:** move the common cutoff to the beginning of the selected interval. Which
values can still be available? For the default sample, the snapshot arrives one
microsecond after the interval begins; it cannot appear early. The following cell is an
answer scaffold that changes the query, not the saved evidence.


In [7]:
earlier_cutoff = min(item.query.start for item in selection.inputs)
earlier_selection = ResearchSelection(
    inputs=tuple(
        replace(item, query=replace(item.query, known_at=earlier_cutoff))
        for item in selection.inputs
    )
)
reader = ResearchReader(
    data_directory / "snapshots", RawStore(data_directory / "raw"), readers=SourceAdapters()
)
earlier = reader.read(earlier_selection)
earlier_counts = [
    (
        item.name,
        sum(
            isinstance(row, (BookMarketRow, QueryMarketRow))
            for batch in item.batches
            for row in batch.rows
        ),
    )
    for item in earlier.inputs
]
display(pd.DataFrame(earlier_counts, columns=["Input", "Earlier market rows"]))
if synthetic:
    assert all(count == 0 for _, count in earlier_counts)
assert reader.read(selection) == dataset, "Earlier inspection changed the pinned result"
print("Earlier selection:", earlier.identity)
print("Original selection reproduced:", dataset.identity)

,Input,Earlier market rows
0,btc,0
1,eth,0
2,okx_depth,0


Earlier selection: e0f4faba2f53f099d271efb9a4bf589e84fcfbe81efc0816329e0b1150707bb8
Original selection reproduced: 0a6c78bc4b2d4fc22f0d8a7f6a91d40cc5427362d31b487c77eac4876f6433a1


Corrections and historical membership also depend on when they became known. A final
revised value must not be used at every earlier decision. The
[installed cutoff example](../tests/integration/consumers/query/asof_consumer.py) retains
an unknown original, a correction available at second 10 and a membership withdrawal at
noon; [access regressions](../tests/unit/data/test_research_access.py) cover those boundaries.
The engineer notebook runs the existing cutoff consumer. Changing a query cannot repair
missing source evidence.


## 4. Prototype a feature and specify its label separately

**Book prototype: exploratory only.** The next section uses the implemented versioned
trade-price feature/label APIs; book features remain conditional.
For the default synthetic L2 state, calculate visible-size imbalance
`(bid BTC - ask BTC) / (bid BTC + ask BTC)`. This is dimensionless and uses only the two
selected sides at the row's original receipt. Keep decimal arithmetic and attach its
state/selection identity and degraded quality. A zero denominator means unavailable.

The calculation is deliberately restricted to the named synthetic teaching input.
For a saved study, first supply the appropriate formula, units, stale-state and missingness
policy; the notebook will not apply an assumed BTC formula to arbitrary data.
A later return label needs a separately defined future horizon, price convention and
label-availability interval. It cannot enter an earlier feature vector.

In [8]:
prototype = []
if synthetic:
    depth = next(item for item in dataset.inputs if item.name == "okx_depth")
    assert isinstance(depth, BookResearchInput)
    for batch in depth.batches:
        assert batch.columns == ("bids", "asks")
        for row in batch.rows:
            if not isinstance(row, BookMarketRow):
                continue
            sizes = []
            for side in row.values:
                assert all(
                    isinstance(level, tuple) and isinstance(level[1], Level) for level in side
                )
                sizes.append(sum((level[1].size for level in side), Decimal(0)))
            total = sizes[0] + sizes[1]
            imbalance = None if total == 0 else (sizes[0] - sizes[1]) / total
            prototype.append(
                {
                    "received_ns": row.received_ns,
                    "state": row.state_identity,
                    "bid_BTC": str(sizes[0]),
                    "ask_BTC": str(sizes[1]),
                    "imbalance": str(imbalance),
                    "quality": row.status,
                }
            )
    assert len(prototype) == 1 and prototype[0]["imbalance"] == "-0.2"
    display(pd.DataFrame(prototype))
else:
    print("Prototype not applied: define a study-specific formula and unit/quality policy first.")

,received_ns,state,bid_BTC,ask_BTC,imbalance,quality
0,1717200000000001000,0027d937a22fbbf77271305be379fc7a38cacdc4520171e1d33c0c7955ccd740,2,3,-0.2,degraded


## 5. Publish a controlled trade study with explicit availability

The earlier unknown-availability archives and book prototype cannot supply this model or
backtest. This **new synthetic BTCUSDT study** contains six minutes of authored prices,
three prints per minute, and explicit correction availability. It is the same saved snapshot
for all subsequent feature, model and economic cells. Membership is the existing synthetic
fixture. Receipt is one day later; the authored correction clock is deliberate software
evidence, never a claim about real Binance availability.

The package publishes and reopens raw ZIP bytes, provenance and quality. Only the tiny
teaching fixture is authored here; query selection, calculations, fitting, risk and accounting
stay in their functional libraries. Re-running a preparation cell refuses its existing directory.

In [9]:
# standard library imports
from contextlib import nullcontext
from datetime import UTC, datetime, timedelta
from unittest.mock import patch
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

# project library imports
from applications.backtest import compare_vectorbt, run_backtest
from applications.explore import evaluate_model, inspect_experiment, publish_experiments
from configuration import Configuration
from financial_units import ExactRatio
from market_data.acquisition import AcquisitionManifest
from market_data.contracts import SourceRequest
from market_data.datasets import DatasetPublication
from market_data.query import QueryReader
from market_data.storage import RawFiles
from market_data.validation import Correction, QualityPolicy, TradeRevision
from reference_data.catalogs import CatalogSnapshot
from reference_data.eligibility import EligibilityAssertions
from research.experiments import AttemptStore, TemporalSplit, TimeWindow
from research.studies import EvaluationRecord
from research.studies import ClearMLDestination, ClearMLPublication
from research.exploration import EdaQuery, FeatureStudy
from research.features import TrailingReturn, WindowPolicy
from research.labels import ForwardReturn
from research.models import LinearDefinition, ModelDataset, ModelStudy, ModelVersion, RetainedResult
from simulation import BacktestSpec, EconomicCheckpoint, EconomicResult, VectorbtAdapter

price_directory = local / "price-study"
price_directory.mkdir()
(price_directory / "raw").mkdir()
(price_directory / "snapshots").mkdir()
start = datetime(2024, 6, 1, tzinfo=UTC)
end = start + timedelta(minutes=6)
catalog = codec.decode_json(
    (root / "configs/reference/binance-archive-spot.json").read_bytes(), CatalogSnapshot
)
job = Configuration.read(root / "configs/data-jobs/binance-archive.json").resolve(
    "spot-comparison", catalog, implementation_identity=ImplementationIdentity.sha256()
)
job = replace(job, instruments=(job.instruments[0],))
request = SourceRequest.partition(job)[0]
prices_by_minute = (
    ("100", "101", "102"),
    ("102", "103", "104"),
    ("105", "104", "103"),
    ("102", "103", "104"),
    ("106", "105", "104"),
    ("103", "102", "101"),
)
prints = tuple(
    (minute * 60 + offset, price)
    for minute, prices in enumerate(prices_by_minute)
    for offset, price in zip((1, 30, 50), prices, strict=True)
)
wire = b"".join(
    f"{trade_id},{value},2,2,{1717200000000 + second * 1000},true,true\n".encode()
    for trade_id, (second, price) in enumerate(prints)
    for value in ("1", price)
)
archive_path = price_directory / "synthetic.zip"
member = ZipInfo("BTCUSDT-trades-2024-06-01.csv")
member.compress_type = ZIP_DEFLATED
with ZipFile(archive_path, "w") as archive:
    archive.writestr(member, wire)
origin = RawFiles.describe(
    archive_path,
    request=request.binding,
    retrieval_reference="synthetic:notebook-price-study",
    ts_received=start + timedelta(days=1),
    ts_ingested=start + timedelta(days=1),
)
raw = RawStore(price_directory / "raw")
raw.publish(archive_path, descriptor=origin, request=request.binding)
acquisition = AcquisitionManifest(run=job, completed=(origin,))
publisher = DatasetPublication(SourceAdapters(), ImplementationIdentity.sha256())
occurrences = tuple(publisher.occurrences(acquisition, raw, max_bytes=2_000_000))
corrections = tuple(
    Correction(
        supersedes=TradeRevision(occurrences=(occurrences[2 * index],)).identity,
        replacement=TradeRevision(occurrences=(occurrences[2 * index + 1],)).identity,
        ts_published=start + timedelta(seconds=second),
        ts_available=start + timedelta(seconds=second),
        evidence_reference="synthetic:notebook-authored-availability",
    )
    for index, (second, _) in enumerate(prints)
)
snapshot_id = publisher.publish(
    (acquisition,),
    raw,
    price_directory / "snapshots",
    max_bytes=2_000_000,
    policy=QualityPolicy(use="exploratory", known_at=start, max_gap=timedelta(hours=1)),
    eligibility=EligibilityAssertions.read(
        root / "tests/fixtures/eligibility/membership-corrections.json", catalog=catalog
    ),
    corrections={job.identity: corrections},
)
binding = (request.instrument.source, "trade", request.instrument.instrument_id)
price_selection = ResearchSelection(
    inputs=(
        TradeSelection(
            name="prices",
            query=EdaQuery(
                snapshot_id=snapshot_id,
                precedence=(binding,),
                instruments=(binding[2],),
                start=start,
                end=end + timedelta(seconds=1),
                known_at=end + timedelta(seconds=1),
                columns=("revision",),
                mode="cutoff",
                batch_size=1,
                max_bytes=2_000_000,
            ),
        ),
    )
)
(price_directory / "selection.json").write_bytes(codec.encode_json(price_selection))
print("Synthetic snapshot:", snapshot_id, "; authored prints:", len(prints))
print("Retained study:", price_directory)

Synthetic snapshot: 9b97851b236a2b7ec42a63b0c7dd419753c87dc7820783d390f2fd92b9d4b723 ; authored prints: 18
Retained study: /Users/lxg/.codex/worktrees/mr-07-research-comparison/alpha-research-lab-private/examples/runs/research-foundations/dfd2e123-d034-4e5d-8c6d-a012fc764dcc-artifacts/price-study


## 6. Retain versioned features and separate forward labels

Use a 60-second trailing window, at least two trades, a 15-second maximum final-price age,
and a 60-second cadence. The forward outcome has a separate 60-second horizon and a
one-second publication delay. Features never receive those future labels. The sample at
minute one should have feature `102/100 − 1 = 1/50` and label `104/102 − 1 = 1/51`.

`inspect_experiment` retains exact configuration, run identity, every attempt and result.
Its diagnostic split is no-fit; the model's narrower training/validation split comes next.
The interval starting at minute eight is reserved and never opened. Repeat the same request
to distinguish reproducible **run identity** from unique **attempt identity**.

In [10]:
diag_split = TemporalSplit(
    training=None,
    fit_cutoff=None,
    fit_mode="none",
    samples=(),
    embargo_us=0,
    validation=TimeWindow(start=start, end=end + timedelta(seconds=2)),
    holdout=TimeWindow(start=start + timedelta(minutes=8), end=start + timedelta(minutes=9)),
    rationale="Authored synthetic diagnostic; reserved future holdout is never read.",
)
feature_study = FeatureStudy(
    family="notebook-price-diagnostics",
    question=question["connected_study"],
    selection=price_selection,
    partition="validation",
    split=diag_split,
    decisions=TimeWindow(start=start + timedelta(minutes=1), end=start + timedelta(seconds=301)),
    feature=TrailingReturn(
        policy=WindowPolicy(lookback_us=60_000_000, min_trades=2, max_staleness_us=15_000_000),
        cadence_us=60_000_000,
        processing_delay_us=0,
    ),
    label=ForwardReturn(
        horizon_us=60_000_000,
        availability_delay_us=1_000_000,
        policy=WindowPolicy(lookback_us=60_000_000, min_trades=1, max_staleness_us=15_000_000),
    ),
)
feature_path = price_directory / "features.json"
feature_path.write_bytes(codec.encode_json(feature_study))
attempts = price_directory / "attempts"
features = inspect_experiment(feature_path, price_directory / "snapshots", raw, attempts=attempts)
repeated = inspect_experiment(feature_path, price_directory / "snapshots", raw, attempts=attempts)
feature_payload = features.events[-1].result
assert feature_payload is not None and repeated.events[-1].result == feature_payload
assert features.start.run_identity == repeated.start.run_identity
assert features.start.attempt_id != repeated.start.attempt_id
model_dataset = ModelDataset.from_results((feature_payload,))
assert model_dataset.rows[0].feature == ExactRatio(numerator="1", denominator="50")
assert model_dataset.rows[0].label == ExactRatio(numerator="1", denominator="51")
display(
    pd.DataFrame(
        [
            {
                "decision": row.sample.decision_at,
                "feature": str(row.feature),
                "label": str(row.label),
                "feature_available": row.sample.features_available_at,
                "label_available": row.sample.label_available_at,
            }
            for row in model_dataset.rows
        ]
    )
)
print(
    "Same run / distinct attempts:",
    features.start.run_identity,
    features.start.attempt_id,
    repeated.start.attempt_id,
)

,decision,feature,label,feature_available,label_available
0,2024-06-01 00:01:00+00:00,"ExactRatio(numerator='1', denominator='50')","ExactRatio(numerator='1', denominator='51')",2024-06-01 00:01:00+00:00,2024-06-01 00:02:01+00:00
1,2024-06-01 00:02:00+00:00,"ExactRatio(numerator='1', denominator='51')","ExactRatio(numerator='-1', denominator='104')",2024-06-01 00:02:00+00:00,2024-06-01 00:03:01+00:00
2,2024-06-01 00:03:00+00:00,"ExactRatio(numerator='-2', denominator='105')","ExactRatio(numerator='1', denominator='103')",2024-06-01 00:03:00+00:00,2024-06-01 00:04:01+00:00
3,2024-06-01 00:04:00+00:00,"ExactRatio(numerator='1', denominator='51')","ExactRatio(numerator='0', denominator='1')",2024-06-01 00:04:00+00:00,2024-06-01 00:05:01+00:00
4,2024-06-01 00:05:00+00:00,"ExactRatio(numerator='-1', denominator='53')","ExactRatio(numerator='-3', denominator='104')",2024-06-01 00:05:00+00:00,2024-06-01 00:06:01+00:00


Same run / distinct attempts: 726b4920d7f4c496aa27144a1b73f7bf3dccf17df4b4df3327ffcf5d856f2891 b125be0c5f674ef7839e15830a345142 20006321c6654ed9ab72e075ea2cce05


## 7. Fit only the training interval; evaluate and reopen retained state

Train on the minute-one and minute-two samples, whose delayed labels are available by
second 181. Declare model availability at second 182; evaluate only minute five.
Minutes three and four are unused; their information intervals separate fitting from validation. Compare train-only exact least squares against a configured zero
predictor. The fixed MSE threshold of 100 is a software exercise, not model acceptance for
investment. All features, labels, coefficients and errors remain exact ratios.

Formal `evaluate_model` requires a ClearML destination. **Offline default:** inject a named
publication failure after real computation/retention, then inspect the terminal attempt.
A completed local result is not a publication receipt. No network call occurs. To perform
real publication deliberately, set `ALPHA_NOTEBOOK_TRACKING_CONFIG` to a non-secret
`ClearMLDestination` JSON and configure SDK credentials externally; failure then propagates.
No notebook starts Docker, supplies credentials or silently switches tracking modes.

In [11]:
tracking_path = os.environ.get("ALPHA_NOTEBOOK_TRACKING_CONFIG")
tracking = (
    codec.decode_json(Path(tracking_path).expanduser().read_bytes(), ClearMLDestination)
    if tracking_path is not None
    else ClearMLDestination(
        project="notebook-software-study",
        api_uri="http://127.0.0.1:18008",
        files_uri="http://127.0.0.1:18081",
    )
)
model_split = TemporalSplit(
    training=TimeWindow(start=start, end=start + timedelta(seconds=121)),
    fit_cutoff=start + timedelta(seconds=181),
    fit_mode="labels",
    validation=TimeWindow(start=start + timedelta(seconds=300), end=start + timedelta(seconds=301)),
    holdout=diag_split.holdout,
    samples=tuple(row.sample for row in model_dataset.rows),
    embargo_us=0,
    rationale="Two synthetic training rows; disjoint validation, no holdout access.",
)
base_model_study = ModelStudy(
    family="notebook-trained-model",
    question="Does exact trained state reproduce its validation errors?",
    sources=(
        RetainedResult(
            attempt_id=features.start.attempt_id, sha256=hashlib.sha256(feature_payload).hexdigest()
        ),
    ),
    split=model_split,
    partition="validation",
    known_at=end + timedelta(seconds=1),
    max_mse=ExactRatio(numerator="100", denominator="1"),
    selection_reason="Frozen software threshold, not empirical model selection",
    definition=LinearDefinition(
        mode="least-squares", assumptions="One synthetic instrument and exact returns"
    ),
    available_at=start + timedelta(seconds=182),
)
configured_study = replace(
    base_model_study,
    family="notebook-zero-predictor",
    split=replace(model_split, training=None, fit_cutoff=None, fit_mode="none"),
    definition=LinearDefinition(
        mode="configured",
        assumptions="Declared zero-return comparator; no fit",
        intercept=ExactRatio(numerator="0", denominator="1"),
        slope=ExactRatio(numerator="0", denominator="1"),
    ),
    available_at=start,
)

In [12]:
model_histories = []
for study in (base_model_study, configured_study):
    config_path = price_directory / f"{study.family}.json"
    config_path.write_bytes(codec.encode_json(study))
    previous_ids = {h.start.attempt_id for h in AttemptStore(attempts).histories()}
    transport = (
        nullcontext()
        if tracking_path is not None
        else patch.object(
            ClearMLPublication,
            "publish",
            side_effect=ConnectionError("offline notebook publication fixture"),
        )
    )
    try:
        with transport:
            evaluate_model(config_path, attempts=attempts, tracking=tracking)
    except ConnectionError as error:
        if tracking_path is not None or str(error) != "offline notebook publication fixture":
            raise
    created = [
        h for h in AttemptStore(attempts).histories() if h.start.attempt_id not in previous_ids
    ]
    assert len(created) == 1 and created[0].status in ("completed", "rejected"), [
        (h.status, h.events[-1].reason) for h in created
    ]
    model_histories.append(created[0])
store = AttemptStore(attempts)
evaluations = [
    codec.decode_json(store.artifact(h.start.attempt_id, "evaluation"), EvaluationRecord)
    for h in model_histories
]
model_versions = [
    codec.decode_json(store.artifact(h.start.attempt_id, "model-version"), ModelVersion)
    for h in model_histories
]
assert len(model_versions[0].fit_rows) == 2 and not model_versions[1].fit_rows
display(
    pd.DataFrame(
        [
            {
                "family": h.start.spec.family,
                "status": h.status,
                "fit_rows": len(v.fit_rows),
                "scored_rows": len(e.predictions),
                "MSE_exact": str(e.mean_squared_error),
                "publication": "acknowledged"
                if "clearml-receipt" in store.artifacts(h.start.attempt_id)
                else "PENDING — injected offline failure",
            }
            for h, v, e in zip(model_histories, model_versions, evaluations, strict=True)
        ]
    )
)

,family,status,fit_rows,scored_rows,MSE_exact,publication
0,notebook-trained-model,completed,2,1,"ExactRatio(numerator='160237688209', denominator='19755989136')",PENDING — injected offline failure
1,notebook-zero-predictor,completed,0,1,"ExactRatio(numerator='9', denominator='10816')",PENDING — injected offline failure


### Reload without refitting; keep a recoverable publication outbox

Reload the exact retained model manifest and state, verify its identity, and reproduce
validation predictions. This is the library inference boundary; the developer notebook also
checks it in a fresh installed process. Once a server is explicitly selected, retry only
`publish_experiments(attempts)` to drain the outbox rather than rerunning model construction.

In [13]:
trained = model_versions[0]
restored = codec.decode_json(
    store.artifact(model_histories[0].start.attempt_id, "model-version"), ModelVersion
)
assert restored.identity == trained.identity
prediction_by_row = {
    row_id: prediction for row_id, _, prediction, _, _ in evaluations[0].predictions
}
for row in model_dataset.rows:
    if row.sample.row_id in prediction_by_row:
        assert (
            restored.predict(row, model_dataset.signature) == prediction_by_row[row.sample.row_id]
        )
if tracking_path is not None:
    before_publish = store.histories()
    receipts = publish_experiments(attempts)
    assert publish_experiments(attempts) == receipts and store.histories() == before_publish
    print("Acknowledged publication receipts:", len(receipts))
else:
    print("SKIPPED real ClearML publication: no explicit tracking configuration. Outbox retained.")
print("Exact reload agreement; no fitting on validation or holdout.")

SKIPPED real ClearML publication: no explicit tracking configuration. Outbox retained.
Exact reload agreement; no fitting on validation or holdout.


## 8. Run fixed targets through independent risk, delayed fills and accounting

A fitted model is optional research output. MR 5 currently supports **no-trade** and
**completed-price momentum**; it does not turn the fitted predictions above into orders.
Use the same saved trade snapshot, baseline spot units, 0.001 BTC target, one-minute closes,
one-second delay, adverse price adjustment and half-even fee rounding from the explicit spec.
This six-minute teaching sample is not the frozen one-hour P1 experiment.

The expected momentum path buys at second 121, sells at 181, buys at 241 and sells at 301.
An independent hand calculation gives gross P&L −0.006 USDT, adverse slippage 0.00004,
fees 0.000416 and net P&L **−0.006456 USDT**. A valid loss is still a completed run.
No-trade stays at 1,000 USDT. No terminal liquidation is invented.

In [14]:
price_reader = QueryReader.read(
    price_directory / "snapshots",
    raw,
    readers=SourceAdapters(),
    identity=snapshot_id,
    precedence=(binding,),
    max_bytes=2_000_000,
)
base_spec = codec.decode_json(
    (root / "tests/fixtures/economics/spec.json").read_bytes(), BacktestSpec
)
economic_spec = replace(
    base_spec, instance="notebook-momentum", binding=binding, start=start, end=end
)
(price_directory / "economics.json").write_bytes(codec.encode_json(economic_spec))
no_trade_spec = replace(economic_spec, instance="notebook-no-trade", rule="no-trade-v1")
no_trade = run_backtest(price_reader, no_trade_spec)
economics = run_backtest(price_reader, economic_spec)
assert isinstance(no_trade, EconomicResult) and isinstance(economics, EconomicResult)
assert no_trade.terminal.nav.value == Decimal("1000")
assert economics.terminal.nav.value == Decimal("999.99354400")
assert economics.terminal.net.value == Decimal("-0.00645600")
assert len(economics.checkpoint.state.ledger.fills) == 4
(price_directory / "economic-reference.json").write_bytes(codec.encode_json(economics))
display(
    pd.DataFrame(
        [
            {
                "at": event.at,
                "phase": event.phase,
                "reason": event.reason,
                "target_BTC": None if event.target is None else str(event.target.value),
                "cash_USDT": str(event.balance.cash.value),
                "holding_BTC": str(event.balance.holding.value),
                "NAV_USDT": None if event.mark is None else str(event.mark.nav.value),
            }
            for event in economics.checkpoint.state.trace
        ]
    )
)
print("Exact cost assumptions:", codec.encode_json(economic_spec.costs).decode())
print("No-trade NAV:", no_trade.terminal.nav.value, "; momentum NAV:", economics.terminal.nav.value)

,at,phase,reason,target_BTC,cash_USDT,holding_BTC,NAV_USDT
0,2024-06-01 00:01:00+00:00,decision,warmup,NaN,1000,0,NaN
1,2024-06-01 00:01:00+00:00,mark,valued,NaN,1000,0,1000.000000000000000000
2,2024-06-01 00:02:00+00:00,decision,intent_created,0.00100,1000,0,NaN
3,2024-06-01 00:02:00+00:00,mark,valued,NaN,1000,0,1000.000000000000000000
4,2024-06-01 00:02:01+00:00,fill,filled,NaN,999.894884990000000000,0.001000000000000000,NaN
5,2024-06-01 00:03:00+00:00,decision,intent_created,0,999.894884990000000000,0.001000000000000000,NaN
6,2024-06-01 00:03:00+00:00,mark,valued,NaN,999.894884990000000000,0.001000000000000000,999.997884990000000000
7,2024-06-01 00:03:01+00:00,fill,filled,NaN,999.996773000000000000,0E-18,NaN
8,2024-06-01 00:04:00+00:00,decision,intent_created,0.00100,999.996773000000000000,0E-18,NaN
9,2024-06-01 00:04:00+00:00,mark,valued,NaN,999.996773000000000000,0E-18,999.996773000000000000


Exact cost assumptions: {"adverse":{"currency":{"code":"USDT","minor_unit":"0.00000001"},"per":{"instrument_id":{"uuid_value":"11111111-1111-4111-8111-111111111111"},"kind":"base_unit"},"value":"0.01"},"fee_rate":"0.001","rounding":"half_even"}
No-trade NAV: 1000.000000000000000000 ; momentum NAV: 999.993544000000000000


### Resume the complete economy and compare the bounded vectorbt adapter

Stop after the decision at second 120 while the buy intent is pending. Persist the full
checkpoint, reopen source bytes, and resume with a different query batch size. A source
cursor alone cannot restore the pending reservation, schedule, observations or ledger.

vectorbt independently reconstructs targets and delayed source prints before comparing the
reference outcomes. It has explicit costs, no partial fills or shared capital, and requires
capital covering gross purchases without sale-proceeds reuse. Its float NAV is analytical;
it does not replace the exact ledger, model queue position or prove executable liquidity.

In [15]:
checkpoint = run_backtest(
    price_reader, economic_spec, batch_size=1, stop_at=start + timedelta(seconds=120)
)
assert isinstance(checkpoint, EconomicCheckpoint) and checkpoint.state.pending is not None
checkpoint_path = price_directory / "economic-checkpoint.json"
checkpoint_path.write_bytes(codec.encode_json(checkpoint))
reopened_reader = QueryReader.read(
    price_directory / "snapshots",
    RawStore(price_directory / "raw"),
    readers=SourceAdapters(),
    identity=snapshot_id,
    precedence=(binding,),
    max_bytes=2_000_000,
)
resumed = run_backtest(
    reopened_reader,
    economic_spec,
    batch_size=2,
    checkpoint=codec.decode_json(checkpoint_path.read_bytes(), EconomicCheckpoint),
)
assert codec.encode_json(resumed) == codec.encode_json(economics)
vector = VectorbtAdapter.run(economics)
(price_directory / "vectorbt.json").write_bytes(codec.encode_json(vector))
display(
    pd.DataFrame(
        [
            {"stage": "complete checkpoint resume", "result": "exact byte equality"},
            {"stage": "reference terminal NAV", "result": str(economics.terminal.nav.value)},
            {"stage": "vectorbt terminal NAV (approximate)", "result": repr(vector.nav[-1])},
            {"stage": "engine and versions", "result": str(vector.versions)},
            {
                "stage": "tolerance (quote units / relative)",
                "result": f"{vector.absolute_tolerance} / {vector.relative_tolerance}",
            },
        ]
    )
)

,stage,result
0,complete checkpoint resume,exact byte equality
1,reference terminal NAV,999.993544000000000000
2,vectorbt terminal NAV (approximate),999.993544
3,engine and versions,"(('vectorbt', '1.1.0'), ('numba', '0.67.0'), ('llvmlite', '0.49.0'), ('numpy', '2.4.6'), ('pandas', '3.0.5'), ('plotly', '6.3.1'))"
4,tolerance (quote units / relative),1e-10 / 1e-12


### Retain the formal economic comparison and its publication state

Use the existing application workflow to retain exact and approximate artifacts together.
The offline transport fixture has the same clearly disclosed meaning as the model section:
local completion succeeds; remote publication remains pending. Actual errors are not relabeled.

In [16]:
previous_ids = {h.start.attempt_id for h in store.histories()}
transport = (
    nullcontext()
    if tracking_path is not None
    else patch.object(
        ClearMLPublication,
        "publish",
        side_effect=ConnectionError("offline notebook publication fixture"),
    )
)
try:
    with transport:
        compare_vectorbt(
            price_reader,
            economic_spec,
            family="notebook-fixed-rule-economics",
            split=diag_split,
            attempts=attempts,
            tracking=tracking,
        )
except ConnectionError as error:
    if tracking_path is not None or str(error) != "offline notebook publication fixture":
        raise
comparison = [h for h in store.histories() if h.start.attempt_id not in previous_ids]
assert len(comparison) == 1 and comparison[0].status == "completed"
assert store.artifact(comparison[0].start.attempt_id, "economic-reference") == codec.encode_json(
    economics
)
display(
    pd.DataFrame(
        [
            {
                "family": h.start.spec.family,
                "attempt": h.start.attempt_id,
                "status": h.status,
                "artifacts": store.artifacts(h.start.attempt_id),
            }
            for h in store.histories()
        ]
    )
)

,family,attempt,status,artifacts
0,notebook-price-diagnostics,20006321c6654ed9ab72e075ea2cce05,completed,()
1,notebook-fixed-rule-economics,b0e8fb1f6fa44059bf71e04cb333afa9,completed,"(clearml-destination, economic-reference, vectorbt-comparison)"
2,notebook-price-diagnostics,b125be0c5f674ef7839e15830a345142,completed,()
3,notebook-zero-predictor,b68c525b51a546c1a7c2e8764ad7f7fa,completed,"(clearml-destination, evaluation, model-version, tracked-models)"
4,notebook-trained-model,ef07e065e0a84db6b3586838fa4c31bb,completed,"(clearml-destination, evaluation, model-version, tracked-models)"


## 9. Compare two declared rules without fitting

Run this cell explicitly to create a bounded two-trial Optuna comparison. Each variant uses
the same saved input, report policy and untouched holdout. Momentum and reversal use the
existing independent risk, delayed-fill and exact accounting path; vectorbt is checked
against those outcomes. Every accepted, rejected, failed or pruned attempt stays retained.
Rerunning with the same journal resumes its fixed budget; it cannot silently extend the grid.

The default notebook uses an explicitly unavailable publication fixture. Supplying the
tracking configuration requires actual ClearML acknowledgement. The separate
[marimo example](research-comparison.py) exposes an explicit Run button over these APIs;
display updates only read retained output. Evidently remains blocked by its dependency audit.


In [17]:
# project library imports
from applications.backtest import compare_parameters
from research.studies import ComparisonStudy
from research.studies import BacktestRun
from research.studies import BenchmarkPeriod, ReportPolicy

flat = ExactRatio(numerator="0", denominator="1")
report_policy = ReportPolicy(
    benchmark_name="explicit synthetic flat control",
    benchmark=tuple(
        BenchmarkPeriod(end=start + timedelta(minutes=i), value=flat) for i in range(1, 7)
    ),
    cadence_seconds=60,
    periods_per_year=525_600,
    annual_risk_free=flat,
)
first_variant = BacktestRun(
    family="notebook-rule-comparison",
    snapshot_id=snapshot_id,
    economics=economic_spec,
    report=report_policy,
    split=diag_split,
)
comparison_study = ComparisonStudy(
    name="notebook-two-rules",
    variants=(
        first_variant,
        replace(
            first_variant,
            economics=replace(
                economic_spec, instance="notebook-reversal", rule="completed-price-reversal-v1"
            ),
        ),
    ),
    budget=2,
    seed=23,
    minimum_return=ExactRatio(numerator="-1", denominator="1"),
)
comparison_config = price_directory / "comparison.json"
comparison_config.write_bytes(codec.encode_json(comparison_study))
transport = (
    nullcontext()
    if tracking_path is not None
    else patch.object(
        ClearMLPublication,
        "publish",
        side_effect=ConnectionError("offline notebook publication fixture"),
    )
)
try:
    with transport:
        compare_parameters(
            comparison_config,
            price_directory / "snapshots",
            price_directory / "raw",
            attempts=attempts,
            journal=price_directory / "comparison.log",
            max_bytes=2_000_000,
            tracking=tracking,
        )
except ConnectionError as error:
    if tracking_path is not None or str(error) != "offline notebook publication fixture":
        raise
rule_comparisons = [
    h for h in store.histories() if h.start.spec.family == "notebook-rule-comparison"
]
assert len(rule_comparisons) == 2 and all(h.status == "completed" for h in rule_comparisons)
assert (
    len({store.artifact(h.start.attempt_id, "selection-identity") for h in rule_comparisons}) == 1
)
assert not any(e.status == "holdout_consumed" for h in rule_comparisons for e in h.events)
display(
    pd.DataFrame(
        [
            {
                "attempt": h.start.attempt_id,
                "status": h.status,
                "reason": h.events[-1].reason,
                "exact_objective": store.artifact(h.start.attempt_id, "trial-evaluation").decode(),
            }
            for h in rule_comparisons
        ]
    )
)

[I 2026-09-21 11:51:52,029] A new study created in Journal with name: notebook-two-rules


[I 2026-09-21 11:51:52,370] Trial 0 finished with value: -6.456e-06 and parameters: {'variant': 0}. Best is trial 0 with value: -6.456e-06.


[I 2026-09-21 11:51:52,698] Trial 1 finished with value: 3.772e-06 and parameters: {'variant': 1}. Best is trial 1 with value: 3.772e-06.


,attempt,status,reason,exact_objective
0,6f5a2b8547454d54a8f8ba0a662abc29,completed,Accepted validation comparison; no promotion authority,"{""net_return"":{""denominator"":""250000000"",""numerator"":""943""}}"
1,cab793c292064d4fab358786e36fb6f6,completed,Accepted validation comparison; no promotion authority,"{""net_return"":{""denominator"":""125000000"",""numerator"":""-807""}}"


## 10. Hand the retained study to quant development

**QR conclusion:** this authored study demonstrates reproducibility and causal/exact software
contracts. It supports no empirical model or trading claim. Training diagnostics and fixed-rule
economics are separate branches over the same input; no unsupported prediction-to-order bridge
has been invented. The holdout remains untouched.

**QD handoff:** reopen `selection.json`, `features.json`, model requests, the attempt store,
`economics.json` and the complete checkpoint with this installed runtime identity. Use the
[developer notebook](infrastructure-diagnostics.ipynb) for fresh-process, unavailable-publication,
source and accounting diagnostics. Preserve the whole artifact directory and matching environment.
Do not copy notebook arithmetic into a second strategy runner.

**Exercise:** predict what increasing the delay past the next eligible print will change, then
run a separately identified spec with `replace(economic_spec, delay_seconds=15)`. Compare fill
clocks and retained results, without altering an existing attempt. This is sensitivity analysis,
not selection on the final holdout. Actual-source empirical acceptance (MR 8), candidate qualification and trading remain later work.

In [18]:
handoff = {
    "status": "completed synthetic research handoff with bounded rule comparisons",
    "input_selection": price_selection.identity,
    "implementation": identity["implementation_sha256"],
    "feature_run": features.start.run_identity,
    "model_versions": [version.identity for version in model_versions],
    "economic_spec": economic_spec.identity,
    "exact_terminal_NAV_USDT": str(economics.terminal.nav.value),
    "artifact_directory": str(price_directory),
    "holdout": "not accessed",
    "tracking": "explicit SDK publication"
    if tracking_path is not None
    else "offline failure fixture; outbox pending",
    "limits": list(economics.limitations),
    "next": "engineer reproduction; MR 7 remaining diagnostics; MR 8 actual-source acceptance",
}
(price_directory / "model-references.json").write_bytes(
    codec.encode_json(
        tuple(
            RetainedResult(attempt_id=h.start.attempt_id, sha256=v.identity)
            for h, v in zip(model_histories, model_versions, strict=True)
        )
    )
)
(local / "handoff.json").write_bytes(codec.encode_json(handoff))
print(json.dumps(handoff, indent=2))
print("Developer reopening: set ALPHA_RESEARCH_STUDY_DIRECTORY to", price_directory)
print("Retained locally; copy the complete directory for a backed-up research record:", local)

{
  "status": "completed synthetic research handoff with bounded rule comparisons",
  "input_selection": "1f4962b6b9e0576551abd7880034271d6b2bed0d32bed12ae6d5b3c19ec10931",
  "implementation": "a3b2511665cc4f93af87a31760a050120b4928a1adc352b6629ff1f4af8b22e5",
  "feature_run": "726b4920d7f4c496aa27144a1b73f7bf3dccf17df4b4df3327ffcf5d856f2891",
  "model_versions": [
    "d93a69daf484a2eefe9b6e3a62e7864f3716975a947f7782776884f87f9645ff",
    "85eecb055c64d53f9bb06c859874adbd4c6c924508e508b3ebd2030b95bbfbfd"
  ],
  "economic_spec": "44c903f2bb84dd825652dbb7cea5d7f140447a4db0afe5ace2f9cf22ba91dd63",
  "exact_terminal_NAV_USDT": "999.993544000000000000",
  "artifact_directory": "/Users/lxg/.codex/worktrees/mr-07-research-comparison/alpha-research-lab-private/examples/runs/research-foundations/dfd2e123-d034-4e5d-8c6d-a012fc764dcc-artifacts/price-study",
  "holdout": "not accessed",
  "tracking": "offline failure fixture; outbox pending",
  "limits": [
    "catalog units are authored research